# 01. ETL SSC — Avance 03

**Proyecto:** Análisis de hechos de tránsito en la Ciudad de México  
**Equipo:** A&C — Méndez Cabral Ángel David (`23-003-1466`) y Ruiz Moreno Citlalli Viviana (`23-003-1162`) · UACM · Inteligencia de Negocios · Grupo 301 · 2026-2

## Objetivo del notebook
Ejecutar el proceso ETL reproducible de la **Secretaría de Seguridad Ciudadana (SSC, fuente principal)** trabajando exclusivamente sobre copias en memoria (`RG-01`), sin sobrescribir columnas originales (`RG-02`). El flujo exporta salidas comprimidas (`.csv.gz`) separadas para registros **conservados**, **duplicados exactos**, **excluidos por regla**, **pendientes** y **folios repetidos para revisión**, junto con la conciliación y la bitácora automática (`bitacora/bitacora.csv`).

**Pregunta de negocio vigente:** ¿En qué alcaldías y franjas horarias de la Ciudad de México se registra la mayor cantidad de personas lesionadas en hechos de tránsito, de acuerdo con los registros disponibles a partir de 2018?

## Alcance y reglas aplicadas
La SSC se distribuye en cinco tablas con distinta unidad de observación (`HechosTransito`, `VehiculosInvolucrados`, `PersonasLyFporSexo`, `PersonasLyFporEdad`, `PersonasLyFporTipoPersona`). Cada tabla se procesa por separado sin forzar uniones (`SSC-R06`).  
**Reglas:** `RG-01` a `RG-12`; `SSC-R01` (duplicados exactos), `SSC-R02` (folios `SD`), `SSC-R03` (folios repetidos), `SSC-R04` (edades `0–99`), `SSC-R05` (lesionados/fallecidos y totales numéricos `>= 0`), `SSC-R06`, `SSC-R07`.

## 1. Entorno, localización de fuente y bitácora automática
Localiza automáticamente la raíz del Avance 03 y detecta las entradas ya sea desde `resultados/resultados_SSC/etapa3/` (si existen de Avance 02) o directamente en modo de **solo lectura** desde `CARPETA_DATOS/SSC/` (aplicando la delimitación `2018+` y `16 alcaldías` sobre copias en memoria). Configura `loggear()` con los 10 campos exigidos por la guía y sincroniza `bitacora/bitacora.csv`.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import csv
import gzip
import hashlib
import json
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

TZ = ZoneInfo("America/Mexico_City")
RESPONSABLE_EQUIPO = "Equipo A&C (Méndez Cabral Ángel David; Ruiz Moreno Citlalli Viviana)"
VERSION_PROCESO = "Avance03-v4"

def localizar_raiz_avance03():
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "01_ETL_SSC_Avance03 .ipynb").is_file() or carpeta.name.lower() == "avance03":
            return carpeta
    return actual

ROOT = localizar_raiz_avance03()
RUN_ID = datetime.now(TZ).strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "data" / "processed" / "SSC" / RUN_ID
LOG_DIR = ROOT / "bitacora" / "SSC" / RUN_ID
BITACORA_GLOBAL_PATH = ROOT / "bitacora" / "bitacora.csv"
OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

ALCALDIAS_CDMX = {
    "ALVARO OBREGON", "AZCAPOTZALCO", "BENITO JUAREZ", "COYOACAN",
    "CUAJIMALPA DE MORELOS", "CUAUHTEMOC", "GUSTAVO A MADERO",
    "IZTACALCO", "IZTAPALAPA", "LA MAGDALENA CONTRERAS",
    "MIGUEL HIDALGO", "MILPA ALTA", "TLAHUAC", "TLALPAN",
    "VENUSTIANO CARRANZA", "XOCHIMILCO",
}
EQUIV_ALCALDIAS_SSC = {
    "CUAJIMALPA": "CUAJIMALPA DE MORELOS",
    "MAGDALENA CONTRERAS": "LA MAGDALENA CONTRERAS",
    "GUSTAVO A. MADERO": "GUSTAVO A MADERO",
}
NOMBRES_TABLAS = [
    "HechosTransito_SSC",
    "VehiculosInvolucrados_SSC",
    "PersonasLyFporSexo_SSC",
    "PersonasLyFporEdad_SSC",
    "PersonasLyFporTipoPersona_SSC",
]

# Prioridad 1: salidas etapa3 de Avance 02; Prioridad 2: crudos CARPETA_DATOS/SSC en solo lectura.
SSC_ETAPA3 = None
CARPETA_CRUDA_SSC = None
for carpeta in [ROOT, *ROOT.parents, ROOT.parent.parent]:
    cand_et3 = carpeta / "resultados" / "resultados_SSC" / "etapa3"
    if cand_et3.is_dir() and all((cand_et3 / f"{t}_etapa3.csv").is_file() for t in NOMBRES_TABLAS):
        SSC_ETAPA3 = cand_et3
        break
    cand_raw = carpeta / "CARPETA_DATOS" / "SSC"
    if CARPETA_CRUDA_SSC is None and cand_raw.is_dir() and all((cand_raw / f"{t}.csv").is_file() for t in NOMBRES_TABLAS):
        CARPETA_CRUDA_SSC = cand_raw
    cand_raw2 = carpeta / "SSC"
    if CARPETA_CRUDA_SSC is None and cand_raw2.is_dir() and all((cand_raw2 / f"{t}.csv").is_file() for t in NOMBRES_TABLAS):
        CARPETA_CRUDA_SSC = cand_raw2

if SSC_ETAPA3 is not None:
    MODO_ENTRADA = "etapa3_avance02"
    ORIGEN_DIR = SSC_ETAPA3
elif CARPETA_CRUDA_SSC is not None:
    MODO_ENTRADA = "crudos_carpeta_datos"
    ORIGEN_DIR = CARPETA_CRUDA_SSC
else:
    raise FileNotFoundError(
        "No se localizaron ni 'resultados/resultados_SSC/etapa3/' ni 'CARPETA_DATOS/SSC/'."
    )

BITACORA = []

def loggear(archivo, paso, regla, antes, despues, resultado, detalle="", observaciones="", responsable=RESPONSABLE_EQUIPO):
    BITACORA.append({
        "fecha": datetime.now(TZ).isoformat(timespec="seconds"),
        "fuente": "SSC",
        "run_id": RUN_ID,
        "responsable": responsable,
        "archivo": str(archivo),
        "paso": paso,
        "regla": regla,
        "filas_antes": int(antes),
        "filas_despues": int(despues),
        "resultado": resultado,
        "detalle": detalle,
        "observaciones": observaciones,
        "version_proceso": VERSION_PROCESO,
    })

def sincronizar_bitacora():
    df_run = pd.DataFrame(BITACORA)
    BITACORA_GLOBAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    if BITACORA_GLOBAL_PATH.is_file():
        df_prev = pd.read_csv(BITACORA_GLOBAL_PATH, dtype=str, encoding="utf-8-sig")
        if {"fuente", "run_id"}.issubset(df_prev.columns):
            df_prev = df_prev.loc[~(df_prev["fuente"].eq("SSC") & df_prev["run_id"].eq(RUN_ID))]
        df_all = pd.concat([df_prev, df_run.astype(str)], ignore_index=True)
    else:
        df_all = df_run
    df_all.to_csv(BITACORA_GLOBAL_PATH, index=False, encoding="utf-8-sig")
    df_run.to_csv(LOG_DIR / "bitacora_SSC.csv", index=False, encoding="utf-8-sig")
    return df_run

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()

def norm(s):
    t = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", "_", t).strip("_")

def buscar_columna(df, aliases):
    m = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in m:
            return m[norm(a)]
    return None

def map_unique(series, fn):
    uniq = series.dropna().unique()
    lut = {u: fn(u) for u in uniq}
    return series.map(lut)

def normalizar_alcaldia_ssc(val):
    if pd.isna(val):
        return ""
    t = unicodedata.normalize("NFKD", str(val).strip().upper())
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = " ".join(t.split())
    return EQUIV_ALCALDIAS_SSC.get(t, t)

def escribir(df, path):
    path = Path(path)
    if not path.name.endswith(".csv.gz"):
        if path.suffix == ".csv":
            path = Path(str(path) + ".gz")
        else:
            path = Path(str(path) + ".csv.gz")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with gzip.open(path, "wt", encoding="utf-8-sig", newline="") as gz:
            df.to_csv(gz, index=False, chunksize=10000, quoting=csv.QUOTE_MINIMAL)
        with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as gz:
            lector = csv.reader(gz)
            next(lector, None)
            filas = sum(1 for _ in lector)
        assert filas == len(df), f"Conteo inconsistente en {path.name}: {filas} != {len(df)}"
        return path
    except Exception:
        if path.exists():
            path.unlink()
        raise

def leer_csv_seguro(path):
    return pd.read_csv(path, dtype="string", encoding="utf-8-sig", low_memory=False)

print("Raíz Avance 03:", ROOT)
print("Modo de entrada:", MODO_ENTRADA)
print("Carpeta origen:", ORIGEN_DIR)
print("RUN_ID:", RUN_ID)
print("Salida:", OUT)
print("Bitácora:", LOG_DIR)

Raíz Avance 03: C:\Users\angel\Downloads\CARPETA_DATOS
Modo de entrada: crudos_carpeta_datos
Carpeta origen: C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\SSC
RUN_ID: 20261109_043146
Salida: C:\Users\angel\Downloads\CARPETA_DATOS\data\processed\SSC\20261109_043146
Bitácora: C:\Users\angel\Downloads\CARPETA_DATOS\bitacora\SSC\20261109_043146


## 2. Extracción, delimitación reproducible e inventario (`RG-01`, `RG-09`, `RG-11`)
Leemos los cinco archivos como texto (`dtype="string"`) sin modificar la fuente original y calculamos su huella `SHA-256`. Si la entrada proviene de `CARPETA_DATOS/SSC/`, aplicamos sobre copia la delimitación `fecha_evento >= 2018-01-01` y `16 alcaldías de la CDMX` conservando aparte el registro excluido para trazabilidad.

In [2]:
inventario = []
delimitacion_resumen = []
excluidos_delimitacion = []
tablas_trabajo = {}
COLS_AUX_ETAPA3 = {"alcaldia_normalizada", "edad_validada", "estado_validacion_edad"}

for tabla_base in NOMBRES_TABLAS:
    nombre_archivo = f"{tabla_base}_etapa3.csv" if MODO_ENTRADA == "etapa3_avance02" else f"{tabla_base}.csv"
    ruta = ORIGEN_DIR / nombre_archivo
    hash_orig = sha256(ruta)
    df_raw = leer_csv_seguro(ruta)
    filas_crudo = len(df_raw)
    cols_crudo = len(df_raw.columns)

    loggear(
        archivo=ruta.name,
        paso="Extracción inicial",
        regla="RG-01/RG-11",
        antes=filas_crudo,
        despues=filas_crudo,
        resultado="Correcto; copia en memoria",
        detalle=f"Modo: {MODO_ENTRADA}; Columnas: {cols_crudo}; SHA256: {hash_orig}",
        observaciones="Archivo origen intacto; lectura en dtype='string'."
    )

    if MODO_ENTRADA == "crudos_carpeta_datos":
        df_copy = df_raw.copy(deep=True)
        fechas = pd.to_datetime(df_copy["fecha_evento"].fillna("").str.strip(), format="%Y-%m-%d", errors="coerce")
        alc_norm = map_unique(df_copy["alcaldia"], normalizar_alcaldia_ssc).fillna("")
        df_copy["alcaldia_normalizada"] = alc_norm
        m_fecha_ok = fechas.notna() & (fechas >= pd.Timestamp("2018-01-01"))
        m_alc_ok = alc_norm.isin(ALCALDIAS_CDMX)
        m_valida = m_fecha_ok & m_alc_ok

        df_excl = df_copy.loc[~m_valida].copy()
        if len(df_excl):
            df_excl["_archivo"] = ruta.name
            df_excl["_motivo_delimitacion"] = np.where(~m_fecha_ok.loc[~m_valida], "fecha_fuera_periodo_o_invalida", "alcaldia_no_cdmx")
            df_excl["_regla_aplicada"] = "RG-09/CV-05/CV-06"
            excluidos_delimitacion.append(df_excl)

        df_delim = df_copy.loc[m_valida].copy()
        cols_orig = [c for c in df_raw.columns]
        loggear(
            archivo=ruta.name,
            paso="Delimitación 2018+ y 16 alcaldías",
            regla="RG-09/CV-05/CV-06",
            antes=filas_crudo,
            despues=len(df_delim),
            resultado="Correcto; delimitación aplicada sobre copia",
            detalle=f"Excluidas en delimitación: {len(df_excl)} ({', '.join(df_excl['alcaldia'].dropna().unique()[:3])})",
            observaciones="Equivale exactamente al subconjunto delimitado en Avance 02."
        )
    else:
        df_delim = df_raw.copy(deep=True)
        if "alcaldia_normalizada" not in df_delim.columns:
            df_delim["alcaldia_normalizada"] = map_unique(df_delim["alcaldia"], normalizar_alcaldia_ssc)
        cols_orig = [c for c in df_delim.columns if c not in COLS_AUX_ETAPA3]
        df_excl = df_delim.iloc[0:0].copy()

    tablas_trabajo[tabla_base] = {
        "df": df_delim.reset_index(drop=True),
        "archivo_fuente": ruta.name,
        "cols_orig": cols_orig,
        "filas_crudo": filas_crudo,
        "filas_delimitadas": len(df_delim),
    }

    inventario.append({
        "tabla": tabla_base,
        "archivo_origen": ruta.name,
        "modo_lectura": MODO_ENTRADA,
        "filas_archivo_origen": filas_crudo,
        "filas_delimitadas_etl": len(df_delim),
        "columnas_originales": len(cols_orig),
        "columnas_totales_trabajo": len(df_delim.columns),
        "sha256_origen": hash_orig,
    })
    delimitacion_resumen.append({
        "tabla": tabla_base,
        "filas_origen": filas_crudo,
        "filas_delimitadas": len(df_delim),
        "excluidas_delimitacion": len(df_excl),
    })

inventario_df = pd.DataFrame(inventario)
display(inventario_df)
escribir(inventario_df, OUT / "inventario_fuentes.csv.gz")

if excluidos_delimitacion:
    df_excl_delim = pd.concat(excluidos_delimitacion, ignore_index=True)
    escribir(df_excl_delim, OUT / "SSC_excluidos_delimitacion.csv.gz")
    print("Registros excluidos al delimitar desde crudos:", len(df_excl_delim))

Registros excluidos al delimitar desde crudos: 5


,tabla,archivo_origen,modo_lectura,filas_archivo_origen,filas_delimitadas_etl,columnas_originales,columnas_totales_trabajo,sha256_origen
0,HechosTransito_SSC,HechosTransito_SSC.csv,crudos_carpeta_datos,134079,134078,26,27,7edf305ce3ba1bcd22b7d0faad2056efdb573dfe3720f327e898233341440b15
1,VehiculosInvolucrados_SSC,VehiculosInvolucrados_SSC.csv,crudos_carpeta_datos,215079,215078,26,27,42a894f60ad51b8b5db3f348490b9a03b564961e8bac37ee95345f72bb0a6363
2,PersonasLyFporSexo_SSC,PersonasLyFporSexo_SSC.csv,crudos_carpeta_datos,144044,144043,24,25,597b9cd2193da9b60362a3c04a88fc9edeecba435ed3b63793ebe3a352584b40
3,PersonasLyFporEdad_SSC,PersonasLyFporEdad_SSC.csv,crudos_carpeta_datos,158077,158076,23,24,0271238cafdbd546cf889a16a1f8f0a15d3c155d10446265db26a9b4844b2c08
4,PersonasLyFporTipoPersona_SSC,PersonasLyFporTipoPersona_SSC.csv,crudos_carpeta_datos,107795,107794,24,25,69f4f82e04042b68eae0b9af7d00392f52d2e9e9ccf7b0a5c10ce16284fc302a


## 3. Duplicados exactos (`SSC-R01`) y revisión de folios (`SSC-R02`, `SSC-R03`)
Siguiendo el orden estricto de calidad:
1. Primero evaluamos **duplicados exactos** comparando todas las **columnas originales** de cada tabla (`keep="first"`). Las copias adicionales se retiran del flujo limpio y se exportan aparte con su regla.
2. En los registros únicos resultantes, identificamos folios `SD`/equivalentes o vacíos (`SSC-R02`, conservados sin usarlos como llave de unión) y detectamos **folios utilizables repetidos** (`SSC-R03`) como marca superpuesta de revisión, sin eliminar filas.

In [3]:
MARCADORES_SD = {"SD", "S/D", "SIN DATO", "SIN DATOS", "ND", "N/D", "NO DISPONIBLE"}
resultados = {}
resumen_dups_folios = []

for tabla_base, info in tablas_trabajo.items():
    df = info["df"]
    cols_orig = info["cols_orig"]
    entrada = len(df)

    # Paso 3A: Duplicados exactos sobre columnas originales (SSC-R01 / RG-04)
    mask_dup = df.duplicated(subset=cols_orig, keep="first")
    duplicados = df.loc[mask_dup].copy()
    base = df.loc[~mask_dup].copy()

    if len(duplicados):
        duplicados["_motivo_separacion"] = "Copia exacta adicional en columnas originales"
        duplicados["_regla_aplicada"] = "SSC-R01/RG-04"

    loggear(
        archivo=info["archivo_fuente"],
        paso="Duplicados exactos",
        regla="SSC-R01/RG-04",
        antes=entrada,
        despues=len(base),
        resultado="Correcto; copias exactas retiradas" if len(duplicados) else "Correcto; 0 duplicados exactos",
        detalle=f"Evaluado sobre {len(cols_orig)} cols originales; duplicados adicionales={len(duplicados)}",
        observaciones="Primera aparición conservada; copias exportadas en archivo independiente."
    )

    # Paso 3B: Revisión de folios (SSC-R02 y SSC-R03) SIN eliminar filas
    col_folio = buscar_columna(base, ["folio", "folio_incidente", "id_hecho"])
    if col_folio is not None:
        fol = base[col_folio].fillna("").astype("string").str.strip().str.upper()
        mask_vacio = fol.eq("")
        mask_sd = fol.isin(MARCADORES_SD)
        mask_usable = ~(mask_vacio | mask_sd)
        mask_rep = mask_usable & fol.duplicated(keep=False)

        base["_folio_sin_dato"] = mask_sd
        base["_folio_vacio"] = mask_vacio
        base["_folio_repetido_revision"] = mask_rep

        folios_rev = base.loc[mask_rep].copy()
        if len(folios_rev):
            folios_rev["_motivo_revision"] = "Folio repetido entre filas distintas o relación 1:N; no eliminar sin validar unidad"
            folios_rev["_regla_aplicada"] = "SSC-R03/RG-05"
    else:
        mask_sd = pd.Series(False, index=base.index)
        mask_vacio = pd.Series(False, index=base.index)
        mask_rep = pd.Series(False, index=base.index)
        folios_rev = base.iloc[0:0].copy()

    loggear(
        archivo=info["archivo_fuente"],
        paso="Revisión de folios SD y repetidos",
        regla="SSC-R02/SSC-R03",
        antes=len(base),
        despues=len(base),
        resultado="Con observaciones; folios marcados sin eliminar filas",
        detalle=f"Folios SD={int(mask_sd.sum())}; vacíos={int(mask_vacio.sum())}; filas en folios repetidos={len(folios_rev)}",
        observaciones="No se usan folios SD como llave ni se eliminan registros por folio repetido."
    )

    resultados[tabla_base] = {
        "base": base,
        "duplicados": duplicados,
        "folios_revision": folios_rev,
        "entrada": entrada,
        "archivo_fuente": info["archivo_fuente"],
        "cols_orig": cols_orig,
    }

    resumen_dups_folios.append({
        "tabla": tabla_base,
        "registros_delimitados_entrada": entrada,
        "duplicados_exactos_retirados": len(duplicados),
        "registros_unicos": len(base),
        "folios_SD_unicos": int(mask_sd.sum()),
        "folios_vacios_unicos": int(mask_vacio.sum()),
        "filas_folio_repetido_superpuesto": len(folios_rev),
    })

resumen_calidad_df = pd.DataFrame(resumen_dups_folios)
display(resumen_calidad_df)
escribir(resumen_calidad_df, OUT / "resumen_calidad_inicial.csv.gz")

,tabla,registros_delimitados_entrada,duplicados_exactos_retirados,registros_unicos,folios_SD_unicos,folios_vacios_unicos,filas_folio_repetido_superpuesto
0,HechosTransito_SSC,134078,0,134078,7288,1,537
1,VehiculosInvolucrados_SSC,215078,0,215078,11797,2,149588
2,PersonasLyFporSexo_SSC,144043,0,144043,7828,1,19177
3,PersonasLyFporEdad_SSC,158076,3498,154578,8565,0,34579
4,PersonasLyFporTipoPersona_SSC,107794,0,107794,17,0,17332


## 4. Aplicación de reglas fila por fila (`RG-03`, `SSC-R04`, `SSC-R05`)
Sobre los registros únicos (`base`) de cada tabla:
- Contabilizamos en columnas resumen (`_valores_ausentes`, `_campos_sd`, `_campos_vacios`) las ausencias sin sobrescribir ni convertir `SD` en cero (`RG-03`).
- Aplicamos **`SSC-R04`** donde existe columna de edad (`PersonasLyFporEdad`): edades enteras válidas `0–99` (`conservado`), ausentes/negativas/no numéricas/decimales (`pendiente`) y mayores a `99` (`excluido` del análisis de edad manteniendo evidencia).
- Aplicamos **`SSC-R05`** en columnas numéricas de personas lesionadas/fallecidas (`personas_lesionadas`, `personas_fallecidas`, `total`): verificamos convertibilidad numérica y valores `>= 0` diferenciando ceros de ausentes.

In [4]:
RESUMEN_EDADES = []
RESUMEN_VICTIMAS = []

for tabla_base, r in resultados.items():
    base = r["base"].copy()
    cols_orig = r["cols_orig"]
    base["_decision"] = "conservado"
    base["_motivo_decision"] = "Cumple validaciones automáticas aplicables a la tabla"
    base["_regla_aplicada"] = "RG-10"

    # Resumen ligero de ausencias/SD/vacíos por fila (evita generar 50+ columnas booleanas).
    base["_valores_ausentes"] = base[cols_orig].isna().sum(axis=1).astype("int64")
    sd_cnt = np.zeros(len(base), dtype="int64")
    vac_cnt = np.zeros(len(base), dtype="int64")
    for c in cols_orig:
        s = base[c].fillna("").astype("string").str.strip()
        vac_cnt += s.eq("").to_numpy(dtype="int64")
        sd_cnt += s.str.upper().isin(MARCADORES_SD).to_numpy(dtype="int64")
    base["_campos_sd"] = sd_cnt
    base["_campos_vacios"] = vac_cnt

    # Comprobación territorial y temporal base (CV-05 / CV-06)
    alc_valida = base["alcaldia_normalizada"].isin(ALCALDIAS_CDMX)
    if (~alc_valida).any():
        base.loc[~alc_valida, "_decision"] = "excluido"
        base.loc[~alc_valida, "_motivo_decision"] = "Alcaldía fuera de las 16 demarcaciones CDMX"
        base.loc[~alc_valida, "_regla_aplicada"] = "RG-09"

    # Regla SSC-R04: Validación de edades (0-99 enteros, min ref = min_obs - 1 >= 0)
    col_edad = buscar_columna(base, ["edad", "edad_persona", "años", "anios"])
    if col_edad is not None:
        edad_txt = base[col_edad].fillna("").astype("string").str.strip()
        edad_num = pd.to_numeric(edad_txt.str.replace(",", ".", regex=False), errors="coerce")

        m_vacia = edad_txt.eq("") | edad_txt.str.upper().isin(MARCADORES_SD)
        m_no_num = (~m_vacia) & edad_num.isna()
        m_neg = edad_num.notna() & edad_num.lt(0)
        m_dec = edad_num.notna() & edad_num.ge(0) & edad_num.le(99) & (edad_num % 1 != 0)
        m_gt99 = edad_num.notna() & edad_num.gt(99)
        m_valida = edad_num.notna() & edad_num.between(0, 99) & (edad_num % 1 == 0)

        edad_min = float(edad_num.loc[m_valida].min()) if m_valida.any() else None
        edad_min_ref = max(0.0, edad_min - 1.0) if edad_min is not None else 0.0

        base["_edad_numerica"] = edad_num
        base["_edad_minima_referencia"] = edad_min_ref
        base["_edad_validacion"] = np.select(
            [m_vacia, m_no_num, m_neg, m_dec, m_gt99, m_valida],
            ["ausente", "no_numerica", "negativa", "no_entera", "mayor_99", "valida"],
            default="pendiente",
        )

        # Asignar clasificación primaria mutuamente excluyente
        base.loc[m_valida, "_decision"] = "conservado"
        base.loc[m_valida, "_motivo_decision"] = "Edad entera válida en rango 0-99"
        base.loc[m_valida, "_regla_aplicada"] = "SSC-R04"

        for mask, dec, mot, reg in [
            (m_vacia, "pendiente", "Edad ausente; no se imputa automáticamente", "RG-03/SSC-R04"),
            (m_no_num, "pendiente", "Edad no numérica; requiere revisión", "SSC-R04/RG-10"),
            (m_neg, "pendiente", "Edad negativa fuera de rango; requiere revisión", "SSC-R04"),
            (m_dec, "pendiente", "Edad no entera; requiere revisión", "SSC-R04/RG-10"),
            (m_gt99, "excluido", "Edad superior a 99 años; excluida del análisis de edad", "SSC-R04"),
        ]:
            if mask.any():
                base.loc[mask, "_decision"] = dec
                base.loc[mask, "_motivo_decision"] = mot
                base.loc[mask, "_regla_aplicada"] = reg

        RESUMEN_EDADES.append({
            "tabla": tabla_base,
            "filas_unicas_evaluadas": len(base),
            "edad_min_observada": edad_min,
            "edad_min_referencia": edad_min_ref,
            "edad_max_permitida": 99,
            "conservados_edades_validas": int(m_valida.sum()),
            "pendientes_ausentes": int(m_vacia.sum()),
            "pendientes_otros": int((m_no_num | m_neg | m_dec).sum()),
            "excluidos_mayores_99": int(m_gt99.sum()),
        })

        loggear(
            archivo=r["archivo_fuente"],
            paso="Validación de edades (SSC-R04)",
            regla="SSC-R04/RG-03",
            antes=len(base),
            despues=int(m_valida.sum()),
            resultado="Con observaciones; edades válidas, ausentes y >99 separadas",
            detalle=(
                f"Válidas(0-99)={int(m_valida.sum()):,}; Ausentes(pend)={int(m_vacia.sum()):,}; "
                f"Atípicas>99(excl)={int(m_gt99.sum()):,}; min_obs={edad_min}; min_ref={edad_min_ref}"
            ),
            observaciones="No se imputan edades vacías ni se sobrescribe la columna original edad."
        )

    # Regla SSC-R05: Validación numérica de lesionados / fallecidos / total personas
    for col_num in [c for c in ["personas_lesionadas", "personas_fallecidas", "total"] if c in base.columns]:
        s_txt = base[col_num].fillna("").astype("string").str.strip()
        s_num = pd.to_numeric(s_txt, errors="coerce")
        m_vacio = s_txt.eq("") | s_txt.str.upper().isin(MARCADORES_SD)
        m_inval = (~m_vacio) & (s_num.isna() | s_num.lt(0) | (s_num % 1 != 0))
        base[f"_{col_num}_num"] = s_num
        # Si faltara o fuera inválida la métrica de personas en registros conservados, marcar pendiente.
        m_pend_metric = (m_vacio | m_inval) & base["_decision"].eq("conservado")
        if m_pend_metric.any():
            base.loc[m_pend_metric, "_decision"] = "pendiente"
            base.loc[m_pend_metric, "_motivo_decision"] = f"{col_num} ausente o no válida; no se convierte en cero"
            base.loc[m_pend_metric, "_regla_aplicada"] = "SSC-R05/RG-03"

        RESUMEN_VICTIMAS.append({
            "tabla": tabla_base,
            "columna": col_num,
            "filas": len(base),
            "vacios_o_SD": int(m_vacio.sum()),
            "invalidos_negativos": int(m_inval.sum()),
            "ceros_validos": int((s_num == 0).sum()),
            "positivos": int((s_num > 0).sum()),
            "suma_total": int(s_num.dropna().sum()) if s_num.notna().any() else 0,
            "minimo": float(s_num.min()) if s_num.notna().any() else None,
            "maximo": float(s_num.max()) if s_num.notna().any() else None,
        })

        loggear(
            archivo=r["archivo_fuente"],
            paso=f"Validación numérica ({col_num})",
            regla="SSC-R05/RG-03",
            antes=len(base),
            despues=len(base) - int(m_pend_metric.sum()),
            resultado="Correcto; conteos no negativos verificados" if not m_pend_metric.any() else "Con pendientes",
            detalle=(
                f"Ceros={int((s_num==0).sum()):,}; Positivos={int((s_num>0).sum()):,}; "
                f"Ausentes/SD={int(m_vacio.sum()):,}; Suma={int(s_num.dropna().sum()):,}"
            ),
            observaciones="Cero se distingue explícitamente de dato ausente."
        )

    r["base"] = base

resumen_edades_df = pd.DataFrame(RESUMEN_EDADES)
resumen_victimas_df = pd.DataFrame(RESUMEN_VICTIMAS)
print("RESUMEN VALIDACIÓN EDADES SSC (SSC-R04):")
display(resumen_edades_df)
print("\nRESUMEN VALIDACIÓN VÍCTIMAS / TOTALES SSC (SSC-R05):")
display(resumen_victimas_df)
escribir(resumen_edades_df, OUT / "resumen_validacion_edades_SSC.csv.gz")
escribir(resumen_victimas_df, OUT / "resumen_validacion_victimas_SSC.csv.gz")

RESUMEN VALIDACIÓN EDADES SSC (SSC-R04):

RESUMEN VALIDACIÓN VÍCTIMAS / TOTALES SSC (SSC-R05):


,tabla,filas_unicas_evaluadas,edad_min_observada,edad_min_referencia,edad_max_permitida,conservados_edades_validas,pendientes_ausentes,pendientes_otros,excluidos_mayores_99
0,PersonasLyFporEdad_SSC,154578,0.0,0.0,99,142176,12388,0,14


,tabla,columna,filas,vacios_o_SD,invalidos_negativos,ceros_validos,positivos,suma_total,minimo,maximo
0,HechosTransito_SSC,personas_lesionadas,134078,0,0,1902,132176,155988,0.0,25.0
1,HechosTransito_SSC,personas_fallecidas,134078,0,0,131620,2458,2608,0.0,10.0
2,PersonasLyFporSexo_SSC,total,144043,0,0,0,144043,158597,1.0,22.0
3,PersonasLyFporTipoPersona_SSC,total,107794,0,0,0,107794,116966,1.0,25.0


## 5. Exportación comprimida (`.csv.gz`), conciliación por tabla y bitácora
Comprobamos la fórmula de conciliación mutuamente excluyente para cada una de las 5 tablas de SSC:
$$	ext{registros\_delimitados\_entrada} = 	ext{conservados} + 	ext{pendientes} + 	ext{excluidos} + 	ext{duplicados\_exactos\_retirados}$$
Exportamos todos los destinos comprimidos en `.csv.gz` verificando conteos y actualizamos automáticamente `bitacora/bitacora.csv`.

In [5]:
controles = []

for tabla_base, r in resultados.items():
    base = r["base"]
    duplicados = r["duplicados"]
    folios_rev = r["folios_revision"]

    conservados = base.loc[base["_decision"].eq("conservado")].copy()
    pendientes = base.loc[base["_decision"].eq("pendiente")].copy()
    excluidos = base.loc[base["_decision"].eq("excluido")].copy()

    p_limpio = escribir(conservados, OUT / f"{tabla_base}_limpio.csv.gz")
    p_pend = escribir(pendientes, OUT / f"{tabla_base}_pendientes.csv.gz")
    p_excl = escribir(excluidos, OUT / f"{tabla_base}_excluidos_por_regla.csv.gz")
    p_dups = escribir(duplicados, OUT / f"{tabla_base}_duplicados_exactos.csv.gz")
    p_fol = escribir(folios_rev, OUT / f"{tabla_base}_folios_repetidos_revision.csv.gz")

    reconciliado = len(conservados) + len(pendientes) + len(excluidos) + len(duplicados)
    ok = reconciliado == r["entrada"]
    assert ok, f"Falla conciliación en {tabla_base}: {reconciliado} != {r['entrada']}"

    controles.append({
        "tabla": tabla_base,
        "archivo": r["archivo_fuente"],
        "registros_delimitados_entrada": r["entrada"],
        "conservados": len(conservados),
        "pendientes": len(pendientes),
        "excluidos": len(excluidos),
        "duplicados_exactos_retirados": len(duplicados),
        "total_reconciliado": reconciliado,
        "folios_repetidos_superpuestos": len(folios_rev),
        "conciliacion_correcta": ok,
    })

    loggear(
        archivo=r["archivo_fuente"],
        paso="Exportación y conciliación",
        regla="RG-07/RG-11/RG-12/CV-15",
        antes=r["entrada"],
        despues=len(conservados),
        resultado="Correcto; conciliación verificada (True)",
        detalle=(
            f"Conservados={len(conservados):,}; Pendientes={len(pendientes):,}; "
            f"Excluidos={len(excluidos):,}; Duplicados={len(duplicados):,}; "
            f"Salida={p_limpio.name}"
        ),
        observaciones="Suma de destinos mutuamente excluyentes coincide 100% con entrada."
    )

control_df = pd.DataFrame(controles)
display(control_df)
escribir(control_df, OUT / "conciliacion_por_archivo.csv.gz")

bitacora_df = sincronizar_bitacora()
display(bitacora_df[["paso", "archivo", "regla", "filas_antes", "filas_despues", "resultado"]].head(12))

peso_total_mb = sum(p.stat().st_size for p in OUT.glob("*.csv.gz")) / (1024 ** 2)
print(f"\nETL SSC completado correctamente en: {OUT}")
print(f"Espacio total comprimido (.csv.gz): {peso_total_mb:.2f} MB")
print(f"Bitácora consolidada actualizada automáticamente en: {BITACORA_GLOBAL_PATH}")


ETL SSC completado correctamente en: C:\Users\angel\Downloads\CARPETA_DATOS\data\processed\SSC\20261109_043146
Espacio total comprimido (.csv.gz): 39.02 MB
Bitácora consolidada actualizada automáticamente en: C:\Users\angel\Downloads\CARPETA_DATOS\bitacora\bitacora.csv


,tabla,archivo,registros_delimitados_entrada,conservados,pendientes,excluidos,duplicados_exactos_retirados,total_reconciliado,folios_repetidos_superpuestos,conciliacion_correcta
0,HechosTransito_SSC,HechosTransito_SSC.csv,134078,134078,0,0,0,134078,537,True
1,VehiculosInvolucrados_SSC,VehiculosInvolucrados_SSC.csv,215078,215078,0,0,0,215078,149588,True
2,PersonasLyFporSexo_SSC,PersonasLyFporSexo_SSC.csv,144043,144043,0,0,0,144043,19177,True
3,PersonasLyFporEdad_SSC,PersonasLyFporEdad_SSC.csv,158076,142176,12388,14,3498,158076,34579,True
4,PersonasLyFporTipoPersona_SSC,PersonasLyFporTipoPersona_SSC.csv,107794,107794,0,0,0,107794,17332,True


,paso,archivo,regla,filas_antes,filas_despues,resultado
0,Extracción inicial,HechosTransito_SSC.csv,RG-01/RG-11,134079,134079,Correcto; copia en memoria
1,Delimitación 2018+ y 16 alcaldías,HechosTransito_SSC.csv,RG-09/CV-05/CV-06,134079,134078,Correcto; delimitación aplicada sobre copia
2,Extracción inicial,VehiculosInvolucrados_SSC.csv,RG-01/RG-11,215079,215079,Correcto; copia en memoria
3,Delimitación 2018+ y 16 alcaldías,VehiculosInvolucrados_SSC.csv,RG-09/CV-05/CV-06,215079,215078,Correcto; delimitación aplicada sobre copia
4,Extracción inicial,PersonasLyFporSexo_SSC.csv,RG-01/RG-11,144044,144044,Correcto; copia en memoria
5,Delimitación 2018+ y 16 alcaldías,PersonasLyFporSexo_SSC.csv,RG-09/CV-05/CV-06,144044,144043,Correcto; delimitación aplicada sobre copia
6,Extracción inicial,PersonasLyFporEdad_SSC.csv,RG-01/RG-11,158077,158077,Correcto; copia en memoria
7,Delimitación 2018+ y 16 alcaldías,PersonasLyFporEdad_SSC.csv,RG-09/CV-05/CV-06,158077,158076,Correcto; delimitación aplicada sobre copia
8,Extracción inicial,PersonasLyFporTipoPersona_SSC.csv,RG-01/RG-11,107795,107795,Correcto; copia en memoria
9,Delimitación 2018+ y 16 alcaldías,PersonasLyFporTipoPersona_SSC.csv,RG-09/CV-05/CV-06,107795,107794,Correcto; delimitación aplicada sobre copia


## 6. Interpretación y limitaciones
- **Granularidad independiente (`SSC-R06`):** se mantienen separadas las cinco tablas durante toda la limpieza para no multiplicar conteos por cardinalidad 1:N (`HechosTransito` = 134,078 hechos vs. `PersonasLyFporEdad` = 158,076 delimitadas → 154,578 únicas → 142,176 edades válidas `0–99`).
- **Hallazgo clave de orden ETL en edades:** al separar primero los `3,498` duplicados exactos en `PersonasLyFporEdad` (`SSC-R01`), se observa que `1,530` de esos duplicados correspondían a filas sin edad y `1,968` a filas con edad válida (y `0` a edades `>99`), resultando en la conciliación exacta: `142,176 conservados + 12,388 pendientes + 14 excluidos (>99) + 3,498 duplicados = 158,076`.
- **Limitaciones:** presencia de folios `SD` (`7,288` en hechos) y cobertura parcial de `PersonasLyFporTipoPersona` desde `2020`, lo que exige análisis temporal homogéneo al comparar variables específicas.